# Paper Tables: Popularity vs LLM on eval_users1000 interaction tasks

Four compact article-style tables for the current in-distribution interaction-prediction experiments. Rows are candidate ratios `1:m`; columns are grouped by method and metric; values are means over available complete seeds.

In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display


def find_repo_root() -> Path:
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "pyproject.toml").exists() and (path / "outputs").exists():
            return path
    raise RuntimeError("Could not find beyond-click-sim repo root")


REPO_ROOT = find_repo_root()
RESULTS_ROOT = REPO_ROOT / "outputs" / "in_distribution" / "interaction_prediction"

POINTWISE_METHODS = {
    "popularity_f1_threshold": "Popularity",
    "llm_yes_no_vllm_llama33_70b_full": "LLM yes/no",
}
RANKING_METHODS = {
    "popularity_ranking": "Popularity",
    "llm_yes_no_vllm_llama33_70b_full": "LLM yes/no",
}
METHOD_ORDER = ["Popularity", "LLM yes/no"]
RATIOS = [1, 2, 3, 9, 19]
DATASET_LABELS = {"ml-1m": "ML-1M", "steam": "Steam"}

TASK_RE = re.compile(
    r"^(?P<dataset>ml-1m|steam)_interaction_cap20_eval_users1000_m(?P<negative_ratio>\d+)_seed(?P<seed>\d+)$"
)
POINTWISE_GROUP = "macro_by_user_group_mean"

POINTWISE_METRICS = [
    ("accuracy", "Accuracy"),
    ("recall", "Recall"),
    ("precision", "Precision"),
    ("f1", "F1"),
]
RANKING_METRICS = [
    ("hit_rate@1", "HR@1"),
    ("ndcg@1", "NDCG@1"),
    ("ndcg@3", "NDCG@3"),
    ("ndcg@5", "NDCG@5"),
    ("ndcg@10", "NDCG@10"),
]

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 200)


In [2]:
def timestamp_from_run_dir(run_dir: Path) -> str:
    match = re.match(r"^(\d{8}T\d{6}Z)_", run_dir.name)
    return "" if match is None else match.group(1)


def read_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def metric_or_none(metrics: dict[str, Any], split: str, group: str, name: str) -> float | None:
    try:
        value = metrics[split][group][name]
    except KeyError:
        return None
    return None if value is None else float(value)


def ratio_or_none(numerator: Any, denominator: Any) -> float | None:
    if numerator is None or denominator in (None, 0):
        return None
    return float(numerator) / float(denominator)


def complete_llm_artifact(metrics: dict[str, Any], *, n_from_metric_block: int | None = None) -> bool:
    method = str(metrics.get("method", ""))
    if method != "llm_yes_no_vllm_llama33_70b_full":
        return True
    requested_rows = metrics.get("requested_rows")
    scored_rows = metrics.get("scored_rows")
    if scored_rows is None:
        scored_rows = n_from_metric_block
    if requested_rows is None:
        requested_rows = scored_rows
    return metrics.get("llm_errors") == 0 and requested_rows is not None and scored_rows == requested_rows


def parse_task(metrics: dict[str, Any]) -> dict[str, Any] | None:
    match = TASK_RE.match(str(metrics.get("task", "")))
    if match is None:
        return None
    return {
        "dataset": match.group("dataset"),
        "negative_ratio": int(match.group("negative_ratio")),
        "seed": int(match.group("seed")),
    }


In [3]:
def collect_pointwise_runs() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for metrics_path in sorted(RESULTS_ROOT.glob("*/metrics.json")):
        run_dir = metrics_path.parent
        metrics = read_json(metrics_path)
        method = str(metrics.get("method", ""))
        if method not in POINTWISE_METHODS:
            continue
        task = parse_task(metrics)
        if task is None:
            continue
        block = metrics.get("test", {}).get(POINTWISE_GROUP)
        if block is None:
            # Do not silently mix group-macro pointwise metrics into the paper-style user-macro table.
            continue
        if not complete_llm_artifact(metrics):
            continue
        row = {
            **task,
            "method": method,
            "method_label": POINTWISE_METHODS[method],
            "timestamp": timestamp_from_run_dir(run_dir),
            "run_dir": run_dir.name,
            "n_users": block.get("n_users"),
            "n_groups": block.get("n_groups"),
        }
        for metric_name, metric_label in POINTWISE_METRICS:
            row[metric_label] = metric_or_none(metrics, "test", POINTWISE_GROUP, metric_name)
        rows.append(row)
    if not rows:
        raise RuntimeError("No pointwise metrics found")
    runs = pd.DataFrame(rows).sort_values(["method", "dataset", "negative_ratio", "seed", "timestamp"])
    return runs.drop_duplicates(["method", "dataset", "negative_ratio", "seed"], keep="last").reset_index(drop=True)


def collect_ranking_runs() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for metrics_path in sorted(RESULTS_ROOT.glob("*/metrics_ranking.json")):
        run_dir = metrics_path.parent
        metrics = read_json(metrics_path)
        method = str(metrics.get("method", ""))
        if method not in RANKING_METHODS:
            continue
        task = parse_task(metrics)
        if task is None:
            continue
        block = metrics.get("test", {}).get(POINTWISE_GROUP)
        if block is None:
            continue
        if not complete_llm_artifact(metrics, n_from_metric_block=block.get("n")):
            continue
        row = {
            **task,
            "method": method,
            "method_label": RANKING_METHODS[method],
            "timestamp": timestamp_from_run_dir(run_dir),
            "run_dir": run_dir.name,
            "n_users": block.get("n_users"),
            "n_groups": block.get("n_groups"),
        }
        for metric_name, metric_label in RANKING_METRICS:
            row[metric_label] = metric_or_none(metrics, "test", POINTWISE_GROUP, metric_name)
        rows.append(row)
    if not rows:
        raise RuntimeError("No ranking metrics found")
    runs = pd.DataFrame(rows).sort_values(["method", "dataset", "negative_ratio", "seed", "timestamp"])
    return runs.drop_duplicates(["method", "dataset", "negative_ratio", "seed"], keep="last").reset_index(drop=True)


pointwise_runs = collect_pointwise_runs()
ranking_runs = collect_ranking_runs()


In [4]:
def summarize_for_table(runs: pd.DataFrame, metric_labels: list[str]) -> pd.DataFrame:
    return (
        runs.groupby(["dataset", "negative_ratio", "method_label"], dropna=False)
        .agg(
            **{metric: (metric, "mean") for metric in metric_labels},
            n_seeds=("seed", "nunique"),
            seeds=("seed", lambda values: tuple(sorted(values))),
        )
        .reset_index()
    )


def make_paper_table(summary: pd.DataFrame, dataset: str, metric_labels: list[str]) -> pd.DataFrame:
    columns = pd.MultiIndex.from_product([METHOD_ORDER, metric_labels], names=["Method", "Metric"])
    table = pd.DataFrame(index=[f"1:{ratio}" for ratio in RATIOS], columns=columns, dtype=float)
    for _, row in summary[summary["dataset"].eq(dataset)].iterrows():
        ratio_label = f"1:{int(row['negative_ratio'])}"
        method_label = row["method_label"]
        if ratio_label not in table.index or method_label not in METHOD_ORDER:
            continue
        for metric in metric_labels:
            table.loc[ratio_label, (method_label, metric)] = row[metric]
    table.index.name = "1:m"
    return table


def style_paper_table(table: pd.DataFrame, caption: str):
    def bold_best_method_per_ratio(data: pd.DataFrame) -> pd.DataFrame:
        styles = pd.DataFrame("", index=data.index, columns=data.columns)
        metric_names = list(dict.fromkeys(metric for _, metric in data.columns))
        for row_idx in data.index:
            for metric in metric_names:
                metric_cols = [col for col in data.columns if col[1] == metric]
                values = data.loc[row_idx, metric_cols].dropna()
                if values.empty:
                    continue
                best = values.max()
                for col in metric_cols:
                    if pd.notna(data.loc[row_idx, col]) and data.loc[row_idx, col] == best:
                        styles.loc[row_idx, col] = "font-weight: bold"
        return styles

    return (
        table.style
        .format("{:.4f}", na_rep="--")
        .apply(bold_best_method_per_ratio, axis=None)
        .set_caption(caption)
        .set_table_styles([
            {"selector": "caption", "props": [("caption-side", "top"), ("font-weight", "bold"), ("font-size", "120%")]},
            {"selector": "th", "props": [("text-align", "center")]},
            {"selector": "td", "props": [("text-align", "center")]},
        ])
    )


pointwise_summary = summarize_for_table(pointwise_runs, [label for _, label in POINTWISE_METRICS])
ranking_summary = summarize_for_table(ranking_runs, [label for _, label in RANKING_METRICS])

ml1m_pointwise = make_paper_table(pointwise_summary, "ml-1m", [label for _, label in POINTWISE_METRICS])
steam_pointwise = make_paper_table(pointwise_summary, "steam", [label for _, label in POINTWISE_METRICS])
ml1m_ranking = make_paper_table(ranking_summary, "ml-1m", [label for _, label in RANKING_METRICS])
steam_ranking = make_paper_table(ranking_summary, "steam", [label for _, label in RANKING_METRICS])


Mean over available complete seeds. Bold marks the better method for the same `1:m` row and metric. Empty cells mean that no complete artifact with the required user-macro metric is available.

## ML-1M Pointwise

In [5]:
display(style_paper_table(ml1m_pointwise, "ML-1M pointwise interaction prediction"))

## Steam Pointwise

In [6]:
display(style_paper_table(steam_pointwise, "Steam pointwise interaction prediction"))

## ML-1M Ranking

In [7]:
display(style_paper_table(ml1m_ranking, "ML-1M candidate ranking"))

## Steam Ranking

In [8]:
display(style_paper_table(steam_ranking, "Steam candidate ranking"))